# FMCSA New Motor Carrier Leads

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Mamoo85/open-data-actor-notebooks/blob/main/notebooks/fmcsa-new-carrier-leads.ipynb)

**What:** New motor carrier leads from the official FMCSA census: new USDOT/MC trucking companies with officer names, authority status, filed email and phone. Filter by state and fleet size; deduped, so daily runs only bill new carriers.

**Why:** run the [`fmcsa-new-carrier-leads`](https://apify.com/transparent_meteorite/fmcsa-new-carrier-leads) Apify actor from Python and analyse the result as a pandas DataFrame, with no scraping code to maintain.

> **Disclosure:** I am the author of this actor. It is pay-per-result; the free plan credit on a new Apify account covers a trial run like this one (this notebook uses a very small sample input).

**Privacy:** Carrier legal names, officers, emails and phone numbers are NOT displayed (they can identify individual owner-operators). Only the USDOT number and company-level fields are shown.

## 1. Setup

In [1]:
!pip install -q apify-client pandas

import os, getpass
try:
    from google.colab import userdata
    APIFY_TOKEN = userdata.get('APIFY_TOKEN')          # Colab: add it under the key icon (Secrets)
except Exception:
    APIFY_TOKEN = os.environ.get('APIFY_TOKEN') or getpass.getpass('Apify API token: ')
assert APIFY_TOKEN, 'An Apify API token is required (a free account is enough).'
print('Token loaded.')

Token loaded.



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Input
A deliberately small sample so the run is cheap and quick. See the [actor page](https://apify.com/transparent_meteorite/fmcsa-new-carrier-leads) for every option.

In [2]:
run_input = {'states': [], 'daysBack': 7, 'requireEmail': True, 'maxResults': 5, 'onlyNewSinceLastRun': False}

## 3. Run the actor and read the dataset

In [3]:
from apify_client import ApifyClient

client = ApifyClient(APIFY_TOKEN)
run = client.actor('transparent_meteorite/fmcsa-new-carrier-leads').call(run_input=run_input)
# apify-client v2 returns a dict, v3 a Run object; handle both
get = (lambda k, a: run[k]) if isinstance(run, dict) else (lambda k, a: getattr(run, a))
items = list(client.dataset(get('defaultDatasetId', 'default_dataset_id')).iterate_items())
print('Run status:', get('status', 'status'), '| items:', len(items))

[apify.fmcsa-new-carrier-leads runId:lVpMekc7ZFzOQ4Smj] -> Status: RUNNING, Message: 


[apify.fmcsa-new-carrier-leads runId:lVpMekc7ZFzOQ4Smj] -> 2026-10-13T22:17:43.100Z ACTOR: Pulling container image of build VmYUgWdDqaRZTXYGR from registry.


[apify.fmcsa-new-carrier-leads runId:lVpMekc7ZFzOQ4Smj] -> 2026-10-13T22:17:43.103Z ACTOR: Creating container.


[apify.fmcsa-new-carrier-leads runId:lVpMekc7ZFzOQ4Smj] -> 2026-10-13T22:17:43.160Z ACTOR: Starting container.


[apify.fmcsa-new-carrier-leads runId:lVpMekc7ZFzOQ4Smj] -> 2026-10-13T22:17:43.161Z ACTOR: Running under "LIMITED_PERMISSIONS".


...

[apify.fmcsa-new-carrier-leads runId:lVpMekc7ZFzOQ4Smj] -> 2026-10-13T22:17:44.705Z [fmcsa] Done. Scanned 1000, delivered 5 new leads (email 100%, phone 100%).


[apify.fmcsa-new-carrier-leads runId:lVpMekc7ZFzOQ4Smj] -> 2026-10-13T22:17:44.776Z INFO  [Status message]: Done: 5 items


[apify.fmcsa-new-carrier-leads runId:lVpMekc7ZFzOQ4Smj] -> 2026-10-13T22:17:44.817Z Thanks for using FMCSA New Carrier Leads. Found a bug or need a field? Open an issue on the Issues tab (https://console.apify.com/actors/bgDLqjPS6lM75VcrS/issues) with the run URL and we will reply there.


[apify.fmcsa-new-carrier-leads runId:lVpMekc7ZFzOQ4Smj] -> Status: SUCCEEDED, Message: Done: 5 items


Run status: SUCCEEDED | items: 5


## 4. Tabulate

Carrier legal names, officers, emails and phone numbers are NOT displayed (they can identify individual owner-operators). Only the USDOT number and company-level fields are shown.

In [4]:
import pandas as pd
pd.set_option('display.max_colwidth', 70)
pd.set_option('display.width', 200)

df = pd.DataFrame(items)

# Privacy: remove contact / personal-data columns before anything is displayed
DROP = ["email", "phone", "author", "registeredagent", "contractor", "officer", "linkedin", "facebook", "instagram", "youtube", "tiktok", "socials", "twitter"]
KEEP = ["emailCount", "phoneCount", "emailProvider"]
df = df[[c for c in df.columns if c in KEEP or not any(p in c.lower() for p in DROP)]]
SHOW = [c for c in ["usdot", "city", "state", "power_units", "total_drivers", "carrier_operation_label", "census_status", "registered_date_iso"] if c in df.columns]
df[SHOW]

,usdot,city,state,power_units,total_drivers,carrier_operation_label,census_status,registered_date_iso
0,9912039,BLAINE,MN,1,4,Interstate,Active,2026-10-12
1,6994401,DEBARY,FL,1,1,Interstate,Active,2026-10-12
2,5992091,BRONX,NY,1,2,Interstate,Active,2026-10-12
3,9757818,BAKERSFIELD,CA,1,4,Interstate,Active,2026-10-11
4,9639830,BUTTE CITY,CA,2,2,Interstate,Active,2026-10-11


## 5. A quick analysis

In [5]:
for c in ['power_units','total_drivers']:
    df[c]=pd.to_numeric(df[c],errors='coerce')
print(df['state'].value_counts())
print('Median fleet (power units):',df['power_units'].median())
df['carrier_operation_label'].value_counts(dropna=False)

state
CA    2
MN    1
FL    1
NY    1
Name: count, dtype: int64
Median fleet (power units): 1.0


carrier_operation_label
Interstate    5
Name: count, dtype: int64

## Next steps

- Raise the limits in the input cell to run a larger sample.
- Schedule the actor or call it from your own pipeline: [fmcsa-new-carrier-leads on the Apify Store](https://apify.com/transparent_meteorite/fmcsa-new-carrier-leads).
- Check the Store page for the current pay-per-result price before scaling up.

Generated by `tools/generate.py`.